In [168]:
# --------------- imports -------------- 
import pandas as pd
import numpy as np
from pathlib import Path
import json

from mlxtend.frequent_patterns import apriori, association_rules


In [169]:
def encode_units(x):
    if x <= 0:
        return 0
    else:
        return 1

my_basket = my_basket.fillna(0)
my_basket_sets = (my_basket > 0).astype(int)

my_basket_sets.head(10)

product,Arabic Coffee,Ayran,Baba Ghanoush,Baklava,Beef Shawarma Plate,Beef Shawarma Sandwich,Cheese Mana'eesh,Chicken Shawarma Plate,Chicken Shawarma Sandwich,Falafel Plate,...,Mixed Grill Plate,Mixed Mana'eesh,Muhammara,Sage Tea,Shanklish Salad,Tabbouleh,Tamarind Juice,Turkish Coffee,Warak Dawali,Zaatar Mana'eesh
transaction,,,,,,,,,,,,,,,,,,,,,
100_11,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
101_29,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
102_27,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
103_25,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
104_8,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
105_27,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
106_6,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
107_23,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
108_8,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [170]:
dataset_path = Path("transactions_dataset.json")
ds = pd.read_json(dataset_path)

In [171]:
# transaction feature creatiion. making a transaction = transaction_id + customer_id

ds['transaction'] = ds['transaction_id'].astype(str) + "_" + ds['customer_id'].astype(str)


In [172]:
ds.head()

,transaction_id,transaction_date,sales_outlet_id,customer_id,product_id,quantity,product_category,product,transaction
0,1,2026-01-05,1,1,1,1,Manaeesh,Zaatar Mana'eesh,1_1
1,1,2026-01-05,1,1,30,1,Coffee & Tea,Arabic Coffee,1_1
2,2,2026-01-05,1,2,2,1,Manaeesh,Cheese Mana'eesh,2_2
3,2,2026-01-05,1,2,31,1,Coffee & Tea,Mint Tea,2_2
4,3,2026-01-06,2,3,30,1,Coffee & Tea,Arabic Coffee,3_3


In [173]:
num_of_items_for_each_transaction = ds['transaction'].value_counts().reset_index()
num_of_items_for_each_transaction.head()

,transaction,count
0,6_6,4
1,33_5,4
2,54_18,4
3,107_23,4
4,112_16,4


In [174]:
num_of_items_for_each_transaction[num_of_items_for_each_transaction['count']==1]

,transaction,count
223,3_3,1
224,38_15,1
225,48_6,1
226,120_2,1
227,121_2,1
228,124_22,1
229,143_6,1
230,179_26,1
231,200_28,1
232,207_24,1


In [175]:
valid_transactions = num_of_items_for_each_transaction[num_of_items_for_each_transaction['count']>1]['transaction'].tolist()
valid_transactions[:9]


['6_6',
 '33_5',
 '54_18',
 '107_23',
 '112_16',
 '115_8',
 '116_15',
 '117_3',
 '123_28']

In [176]:
num_of_items_for_each_transaction[num_of_items_for_each_transaction['count']<=1]


,transaction,count
223,3_3,1
224,38_15,1
225,48_6,1
226,120_2,1
227,121_2,1
228,124_22,1
229,143_6,1
230,179_26,1
231,200_28,1
232,207_24,1


In [177]:
ds = ds[ds['transaction'].isin(valid_transactions)]
ds.shape

(599, 9)

In [178]:
ds['product_category'].value_counts()

product_category
Mezze           119
Coffee & Tea    110
Drinks           99
Salads           81
Sweets           60
Plates           49
Manaeesh         41
Sandwiches       40
Name: count, dtype: int64

In [179]:
# highest product popularity.

ds['product'].value_counts()


product
Fattoush Salad               43
Tabbouleh                    38
Arabic Coffee                36
Turkish Coffee               31
Hummus                       26
Ayran                        25
Mint Tea                     23
Jallab                       23
Sage Tea                     20
Baklava                      20
Fresh Juice                  18
Mint Lemonade                17
Warak Dawali                 17
Tamarind Juice               16
Makdous                      15
Knafeh                       14
Halaweh Bil Jibn             13
Maamoul                      13
Baba Ghanoush                12
Muhammara                    12
Beef Shawarma Plate          12
Kibbeh                       12
Cheese Mana'eesh             11
Chicken Shawarma Sandwich    11
Mixed Grill Plate            11
Beef Shawarma Sandwich       11
Foul Mdammas                 10
Labneh                       10
Chicken Shawarma Plate        9
Lahm Bi Ajeen                 9
Kafta Sandwich                9


In [180]:
product_recomendation = ds.groupby(['product', 'product_category']).count().reset_index()


In [181]:
product_recomendation = product_recomendation[["product", "product_category", "transaction_id"]]
product_recomendation = product_recomendation.rename(columns = {"transaction_id":"number_of_transactions" })


In [182]:
product_recomendation.head()

,product,product_category,number_of_transactions
0,Arabic Coffee,Coffee & Tea,36
1,Ayran,Drinks,25
2,Baba Ghanoush,Mezze,12
3,Baklava,Sweets,20
4,Beef Shawarma Plate,Plates,12


Apriori recomendation engine


In [183]:
train_basket = (ds.groupby(['transaction', 'product'])['product'].count().reset_index(name='count'))
train_basket.head()

,transaction,product,count
0,100_11,Beef Shawarma Sandwich,1
1,100_11,Fresh Juice,1
2,101_29,Beef Shawarma Plate,1
3,101_29,Fattoush Salad,1
4,101_29,Tamarind Juice,1


In [184]:
my_basket = train_basket.pivot_table(index="transaction", columns="product", values="count").fillna(0)
my_basket.head()


product,Arabic Coffee,Ayran,Baba Ghanoush,Baklava,Beef Shawarma Plate,Beef Shawarma Sandwich,Cheese Mana'eesh,Chicken Shawarma Plate,Chicken Shawarma Sandwich,Falafel Plate,...,Mixed Grill Plate,Mixed Mana'eesh,Muhammara,Sage Tea,Shanklish Salad,Tabbouleh,Tamarind Juice,Turkish Coffee,Warak Dawali,Zaatar Mana'eesh
transaction,,,,,,,,,,,,,,,,,,,,,
100_11,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
101_29,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
102_27,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
103_25,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
104_8,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [185]:
def encode_units(x):
    if x <= 0:
        return 0
    else:
        return 1

my_basket = my_basket.fillna(0)
my_basket_sets = (my_basket > 0).astype(int)

my_basket_sets.head(10)

product,Arabic Coffee,Ayran,Baba Ghanoush,Baklava,Beef Shawarma Plate,Beef Shawarma Sandwich,Cheese Mana'eesh,Chicken Shawarma Plate,Chicken Shawarma Sandwich,Falafel Plate,...,Mixed Grill Plate,Mixed Mana'eesh,Muhammara,Sage Tea,Shanklish Salad,Tabbouleh,Tamarind Juice,Turkish Coffee,Warak Dawali,Zaatar Mana'eesh
transaction,,,,,,,,,,,,,,,,,,,,,
100_11,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
101_29,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
102_27,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
103_25,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
104_8,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
105_27,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
106_6,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
107_23,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
108_8,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [186]:

frequent_items = apriori(my_basket_sets, min_support=0.02, use_colnames=True)
frequent_items[:5]

c:\Users\Samer\OneDrive\Desktop\code\projects\sufra_chatbot_fullstack_application\backend\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.161435,frozenset({Arabic Coffee})
1,0.112108,frozenset({Ayran})
2,0.053812,frozenset({Baba Ghanoush})
3,0.089686,frozenset({Baklava})
4,0.053812,frozenset({Beef Shawarma Plate})


In [187]:
rules_basket = association_rules(frequent_items, metric="lift", min_threshold=1)
rules_basket.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Baklava}),frozenset({Arabic Coffee}),0.089686,0.161435,0.026906,0.300000,1.858333,1.0,0.012427,1.197950,0.507389,0.120000,0.165241,0.233333
1,frozenset({Arabic Coffee}),frozenset({Baklava}),0.161435,0.089686,0.026906,0.166667,1.858333,1.0,0.012427,1.092377,0.550802,0.120000,0.084565,0.233333
2,frozenset({Lahm Bi Ajeen}),frozenset({Arabic Coffee}),0.040359,0.161435,0.022422,0.555556,3.441358,1.0,0.015906,1.886771,0.739252,0.125000,0.469994,0.347222
3,frozenset({Arabic Coffee}),frozenset({Lahm Bi Ajeen}),0.161435,0.040359,0.022422,0.138889,3.441358,1.0,0.015906,1.114422,0.845989,0.125000,0.102674,0.347222
4,frozenset({Makdous}),frozenset({Arabic Coffee}),0.067265,0.161435,0.022422,0.333333,2.064815,1.0,0.011563,1.257848,0.552885,0.108696,0.204991,0.236111


In [188]:
rules_basket[rules_basket['antecedents'] == {'Arabic Coffee'}].sort_values('confidence', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,frozenset({Arabic Coffee}),frozenset({Baklava}),0.161435,0.089686,0.026906,0.166667,1.858333,1.0,0.012427,1.092377,0.550802,0.120000,0.084565,0.233333
3,frozenset({Arabic Coffee}),frozenset({Lahm Bi Ajeen}),0.161435,0.040359,0.022422,0.138889,3.441358,1.0,0.015906,1.114422,0.845989,0.125000,0.102674,0.347222
5,frozenset({Arabic Coffee}),frozenset({Makdous}),0.161435,0.067265,0.022422,0.138889,2.064815,1.0,0.011563,1.083177,0.614973,0.108696,0.076790,0.236111


In [189]:
with open("menu.json", "r", encoding="utf-8") as f:
    menu = json.load(f)

name_to_id = {item["name_en"]: item["id"] for item in menu}
id_to_category = {item["id"]: item["category"] for item in menu}    

In [190]:
from typing import Any


apriori_output = {}

for _, row in rules_basket.iterrows():
    antecedents = list[Any](row["antecedents"])
    consequents = list[Any](row["consequents"])
    confidence = round(row["confidence"], 4)
    lift = round(row["lift"], 4)

    ant_ids = [name_to_id[n] for n in antecedents if n in name_to_id]
    con_ids = [name_to_id[n] for n in consequents if n in name_to_id]

    if not ant_ids or not con_ids:
        continue

    for ant_id in ant_ids:
        if ant_id not in apriori_output:
            apriori_output[ant_id] = []
        apriori_output[ant_id].append({"itemIds": con_ids, "confidence": confidence, "lift": lift})

for key in apriori_output:
    apriori_output[key] = sorted(apriori_output[key], key=lambda x: x["confidence"], reverse=True)

with open("apriori_recommendations.json", "w", encoding="utf-8") as f:
    json.dump(apriori_output, f, indent=2)



In [194]:
popularity_output = []

for _, row in product_recomendation.iterrows():
    name = row["product"]
    count = int(row["number_of_transactions"])
    item_id = name_to_id.get(name)
    if item_id:
        popularity_output.append({"itemId": item_id, "category": id_to_category.get(item_id), "count": count})

popularity_output = sorted(popularity_output, key=lambda x: x["count"], reverse=True)

with open("popularity_recommendations.json", "w", encoding="utf-8") as f:
    json.dump(popularity_output, f, indent=2)